In [5]:
!pip install ipynb

In [ ]:
# main.py
import tkinter as tk
from tkinter import messagebox, ttk
import re

# Extract variables from styles notebook safely
from ipynb.fs.full.styles import (
    BACKGROUND_COLOR, TEXT_COLOR, PRIMARY_COLOR, SECONDARY_COLOR,
    MUTED_TEXT, ERROR_COLOR, SUCCESS_COLOR,
    FONT_TITLE, FONT_SUBTITLE, FONT_BODY, FONT_BUTTON, SKILL_POOL
)

# Extract operations backend engine
from ipynb.fs.full.data_manager import (
    initialise_file, save_user, validate_login, 
    save_job, get_employer_jobs, get_job_applicants
)
from ipynb.fs.full.job_browsing import EmployeeDashboard

# Assemble component style dictionaries natively to bypass file boundary conflicts
BUTTON_STYLE = {
    "bg": PRIMARY_COLOR,
    "fg": "#FFFFFF",
    "font": FONT_BUTTON,
    "relief": "flat",
    "activebackground": SECONDARY_COLOR,
    "activeforeground": "#FFFFFF",
    "padx": 15,
    "pady": 6,
    "cursor": "hand2"
}

ENTRY_STYLE = {
    "font": FONT_BODY,
    "bg": "#FFFFFF",
    "fg": TEXT_COLOR,
    "insertbackground": TEXT_COLOR,
    "relief": "solid",
    "bd": 1
}

LABEL_STYLE = {
    "bg": BACKGROUND_COLOR,
    "fg": TEXT_COLOR,
    "font": FONT_BODY
}

initialise_file()

root = tk.Tk()
root.title("WorkLink - Integrated Employment Workspace")
root.geometry("850x700")
root.config(bg=BACKGROUND_COLOR)

current_user_id = None
current_user_name = None
current_user_role = None

def logout():
    global current_user_id, current_user_name, current_user_role
    current_user_id = None
    current_user_name = None
    current_user_role = None
    show_role_selection()

def clear_screen():
    for widget in root.winfo_children():
        widget.destroy()

def show_role_selection():
    clear_screen()
    root.geometry("520x520")
    
    title = tk.Label(root, text="Welcome to WorkLink", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=40)
    
    subtitle = tk.Label(root, text="Choose how you would like to use WorkLink", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT)
    subtitle.pack(pady=10)

    emp_btn = tk.Button(root, text="I am an Employer", **BUTTON_STYLE, width=28, height=2, command=lambda: show_login("Employer"))
    emp_btn.pack(pady=15)

    employee_btn = tk.Button(root, text="I am an Employee", **BUTTON_STYLE, width=28, height=2, command=lambda: show_login("Employee"))
    employee_btn.pack(pady=15)

    exit_btn_style = BUTTON_STYLE.copy()
    exit_btn_style["bg"] = ERROR_COLOR
    exit_btn_style["activebackground"] = "#D62828"

    exit_btn = tk.Button(root, text="Exit", **exit_btn_style, width=12, command=root.destroy)
    exit_btn.pack(pady=35)

def show_login(role):
    clear_screen()
    root.geometry("520x550")

    title = tk.Label(root, text=f"{role} Login", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=30)

    tk.Label(root, text="Email", **LABEL_STYLE).pack(anchor="w", padx=100)
    email_entry = tk.Entry(root, width=35, **ENTRY_STYLE)
    email_entry.pack(pady=5)

    tk.Label(root, text="Password", **LABEL_STYLE).pack(anchor="w", padx=100)
    password_entry = tk.Entry(root, width=35, show="*", **ENTRY_STYLE)
    password_entry.pack(pady=5)

    def process_login():
        email = email_entry.get().strip()
        pwd = password_entry.get().strip()

        if not email or not pwd:
            messagebox.showerror("Missing Information", "Please fill in all fields")
            return

        profile = validate_login(email, pwd)
        if profile:
            if profile["role"] != role:
                messagebox.showerror("Login Error", f"This account is registered as '{profile['role']}'.")
                return
            
            global current_user_id, current_user_name, current_user_role
            current_user_id = str(profile["id"])
            current_user_name = profile["name"]
            current_user_role = profile["role"]

            messagebox.showinfo("Login Successful", f"Welcome back, {current_user_name}!")
            
            if current_user_role == "Employer":
                show_employer_dashboard()
            else:
                show_employee_dashboard()
        else:
            messagebox.showerror("Login Failed", "Login Failed.")

    submit_btn = tk.Button(root, text="Login", **BUTTON_STYLE, width=22, command=process_login)
    submit_btn.pack(pady=20)

    reg_lnk = tk.Button(root, text="Don't possess an account? Register here", font=("Helvetica", 10, "underline"), bg=BACKGROUND_COLOR, fg=TEXT_COLOR, bd=0, activebackground=BACKGROUND_COLOR, command=lambda: show_register(role))
    reg_lnk.pack(pady=10)

    back_btn_style = BUTTON_STYLE.copy()
    back_btn_style["bg"] = "#777777"
    back_btn_style["activebackground"] = "#555555"

    back_btn = tk.Button(root, text="Return to Main Menu", **back_btn_style, width=25, command=logout)
    back_btn.pack(pady=15)

def show_register(role):
    clear_screen()
    root.geometry("550x650")

    title = tk.Label(root, text=f"Create {role} Account", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=25)

    tk.Label(root, text="Full Name / Company Name", **LABEL_STYLE).pack(anchor="w", padx=100)
    name_entry = tk.Entry(root, width=35, **ENTRY_STYLE)
    name_entry.pack(pady=5)

    tk.Label(root, text="Email", **LABEL_STYLE).pack(anchor="w", padx=100)
    email_entry = tk.Entry(root, width=35, **ENTRY_STYLE)
    email_entry.pack(pady=5)

    tk.Label(root, text="Password", **LABEL_STYLE).pack(anchor="w", padx=100)
    password_entry = tk.Entry(root, width=35, show="*", **ENTRY_STYLE)
    password_entry.pack(pady=5)

    label_txt = "Industry / Job Field" if role == "Employer" else "Primary Skill"
    tk.Label(root, text=label_txt, **LABEL_STYLE).pack(anchor="w", padx=100)
    
    selected_skill_var = tk.StringVar()
    selected_skill_var.set(SKILL_POOL[0]) 
    
    skill_dropdown = ttk.OptionMenu(root, selected_skill_var, SKILL_POOL[0], *SKILL_POOL)
    skill_dropdown.config(width=32)
    skill_dropdown.pack(pady=5)

    def process_registration():
        name = name_entry.get().strip()
        email = email_entry.get().strip()
        pwd = password_entry.get().strip()
        skill = selected_skill_var.get()

        # 1. Empty Field Check
        if not name or not email or not pwd:
            messagebox.showerror("Registration Error", "Please complete all fields.")
            return

        # 2. Email Format Validation (Regex)
        email_pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
        if not re.match(email_pattern, email):
            messagebox.showerror("Invalid Format", "Please enter a valid email address (e.g., user@domain.com).")
            return

        # 3. Password Strength Validation
        if len(pwd) < 6:
            messagebox.showerror("Weak Password", "Your password must be at least 6 characters long.")
            return
            
        # 4. Name Length Limit (UI Protection)
        if len(name) > 50:
            messagebox.showerror("Length Exceeded", "Name must be 50 characters or less.")
            return

        is_saved = save_user(name, email, pwd, role, skill)
        
        # 5. Handle File Permission Errors (Returned from data_manager)
        if is_saved == "LOCKED":
            messagebox.showerror("File Access Denied", "Cannot save profile. Please ensure 'users.csv' is closed in Excel or other programs.")
        elif is_saved:
            messagebox.showinfo("Registration Successful", "Account created successfully. Please log in.")
            show_login(role)
        else:
            messagebox.showerror("Registration Failed", "An account with this email already exists.")

    submit_btn = tk.Button(root, text="Register", **BUTTON_STYLE, width=25, command=process_registration)
    submit_btn.pack(pady=20)

    login_lnk = tk.Button(root, text="Already have a profile? Login here", font=("Helvetica", 10, "underline"), bg=BACKGROUND_COLOR, fg=TEXT_COLOR, bd=0, activebackground=BACKGROUND_COLOR, command=lambda: show_login(role))
    login_lnk.pack(pady=10)

    back_btn_style = BUTTON_STYLE.copy()
    back_btn_style["bg"] = "#777777"
    back_btn_style["activebackground"] = "#555555"

    back_btn = tk.Button(root, text="Return to Main Menu", **back_btn_style, width=25, command=logout)
    back_btn.pack(pady=15)

def show_employer_dashboard():
    clear_screen()
    root.geometry("780x650")

    title = tk.Label(root, text="Employer Dashboard", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=25)

    welcome = tk.Label(root, text=f"Welcome: {current_user_name}", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=TEXT_COLOR)
    welcome.pack(pady=10)

    post_btn = tk.Button(root, text="Post a New Job", **BUTTON_STYLE, width=35, command=show_post_job_screen)
    post_btn.pack(pady=12)

    view_btn = tk.Button(root, text="View Posted Jobs", **BUTTON_STYLE, width=35, command=show_employer_jobs_screen)
    view_btn.pack(pady=12)

    logout_btn_style = BUTTON_STYLE.copy()
    logout_btn_style["bg"] = ERROR_COLOR
    logout_btn_style["activebackground"] = "#D62828"

    logout_btn = tk.Button(root, text="Logout", **logout_btn_style, width=35, command=logout)
    logout_btn.pack(pady=12)

def show_post_job_screen():
    clear_screen()

    title = tk.Label(root, text="Post New Job", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=20)

    # --- 1. TITLE FIELD WITH DYNAMIC COUNTER ---
    tk.Label(root, text="Job Title", **LABEL_STYLE).pack(anchor="w", padx=150)
    
    title_var = tk.StringVar()
    title_entry = tk.Entry(root, textvariable=title_var, width=50, **ENTRY_STYLE)
    title_entry.pack(pady=(5, 0)) # Padded slightly less on the bottom to fit the counter

    # Title Counter Label (Anchored to the right side)
    title_counter_lbl = tk.Label(root, text="0/60 characters", font=("Helvetica", 9), bg=BACKGROUND_COLOR, fg=MUTED_TEXT)
    title_counter_lbl.pack(anchor="e", padx=150)

    def update_title_counter(*args):
        current_len = len(title_var.get())
        title_counter_lbl.config(text=f"{current_len}/60 characters")
        if current_len > 60:
            title_counter_lbl.config(fg=ERROR_COLOR) # Turn red if over limit
        else:
            title_counter_lbl.config(fg=MUTED_TEXT)

    # Attach the counter function to trigger every time the user types
    title_var.trace_add("write", update_title_counter)


    # --- 2. DESCRIPTION FIELD WITH DYNAMIC COUNTER ---
    tk.Label(root, text="Job Description", **LABEL_STYLE).pack(anchor="w", padx=150, pady=(10, 0))
    desc_entry = tk.Text(root, width=50, height=5, font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR, relief="solid", bd=1)
    desc_entry.pack(pady=(5, 0))

    # Description Counter Label
    desc_counter_lbl = tk.Label(root, text="0/1000 characters", font=("Helvetica", 9), bg=BACKGROUND_COLOR, fg=MUTED_TEXT)
    desc_counter_lbl.pack(anchor="e", padx=150)

    def update_desc_counter(event):
        # "end-1c" stops Tkinter from counting the hidden newline character it always adds to the end
        current_len = len(desc_entry.get("1.0", "end-1c"))
        desc_counter_lbl.config(text=f"{current_len}/1000 characters")
        if current_len > 1000:
            desc_counter_lbl.config(fg=ERROR_COLOR)
        else:
            desc_counter_lbl.config(fg=MUTED_TEXT)

    # Attach the counter function to trigger every time a keyboard key is released
    desc_entry.bind("<KeyRelease>", update_desc_counter)


    # --- THE REST OF THE SCREEN ---
    tk.Label(root, text="Required Skill", **LABEL_STYLE).pack(anchor="w", padx=150, pady=(10, 0))
    job_skill_var = tk.StringVar()
    job_skill_var.set(SKILL_POOL[0])
    
    job_skill_dropdown = ttk.OptionMenu(root, job_skill_var, SKILL_POOL[0], *SKILL_POOL)
    job_skill_dropdown.config(width=46)
    job_skill_dropdown.pack(pady=5)

    tk.Label(root, text="Experience Level", **LABEL_STYLE).pack(anchor="w", padx=150)
    exp_var = tk.StringVar()
    exp_var.set("Entry-Level")
    exp_menu = tk.OptionMenu(root, exp_var, "No Experience", "Entry-Level", "Intermediate", "Expert")
    exp_menu.config(font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR, width=15)
    exp_menu.pack(pady=8)

    def handle_job_publish():
        t_text = title_var.get().strip()
        d_text = desc_entry.get("1.0", tk.END).strip()
        s_text = job_skill_var.get()
        e_text = exp_var.get()

        if not t_text or not d_text:
            messagebox.showerror("Posting Error", "Job title and description cannot be empty.")
            return

        if len(t_text) > 60:
            messagebox.showerror("Limit Exceeded", "Job title cannot exceed 60 characters.")
            return
            
        if len(d_text) > 1000:
            messagebox.showerror("Limit Exceeded", "Job description cannot exceed 1000 characters.")
            return

        success = save_job(current_user_id, t_text, d_text, s_text, e_text)
        
        if success == "LOCKED":

            messagebox.showerror(
                "File Access Denied",
                "Please close jobs.csv."
                )

        elif success:

            messagebox.showinfo(
                "Success",
                "Job posted successfully."
            )

            show_employer_dashboard()

        else:

            messagebox.showerror(
                "Error",
                "Job could not be saved."
            )

    save_btn = tk.Button(root, text="Post Job", **BUTTON_STYLE, width=25, command=handle_job_publish)
    save_btn.pack(pady=15)

    abort_btn_style = BUTTON_STYLE.copy()
    abort_btn_style["bg"] = "#777777"
    abort_btn_style["activebackground"] = "#555555"
    back_btn = tk.Button(root, text="Cancel", **abort_btn_style, width=25, command=show_employer_dashboard)
    back_btn.pack()
    
def show_employer_jobs_screen():
    clear_screen()

    title = tk.Label(root, text="Your Posted Jobs", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=15)

    canvas = tk.Canvas(root, bg=BACKGROUND_COLOR, highlightthickness=0)
    scrollbar = tk.Scrollbar(root, orient="vertical", command=canvas.yview)
    scroll_frame = tk.Frame(canvas, bg=BACKGROUND_COLOR)

    scroll_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
    canvas.create_window((0, 0), window=scroll_frame, anchor="nw", width=740)
    canvas.configure(yscrollcommand=scrollbar.set)

    canvas.pack(side="left", fill="both", expand=True, padx=20)
    scrollbar.pack(side="right", fill="y")

    my_jobs = get_employer_jobs(current_user_id)

    if not my_jobs:
        tk.Label(scroll_frame, text="No jobs posted yet.", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT).pack(pady=40)
    else:
        for job in my_jobs:
            j_id = job["id"]
            box = tk.Frame(scroll_frame, bg=BACKGROUND_COLOR, bd=1, relief="solid")
            box.pack(fill="x", padx=10, pady=10, ipady=5)

            tk.Label(box, text=job["title"], font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR).pack(anchor="w", padx=15, pady=5)
            tk.Label(box, text=f"Experience Level: {job['experience']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15)
            tk.Label(box, text=f"Required Skill: {job['skills']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15)
            tk.Label(box, text=f"Description: {job['description']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=MUTED_TEXT, wraplength=600, justify="left").pack(anchor="w", padx=15, pady=5)

            review_btn = tk.Button(box, text="View Applicants", **BUTTON_STYLE, command=lambda target_id=j_id, target_title=job["title"]: show_applicants_screen(target_id, target_title))
            review_btn.pack(anchor="e", padx=15, pady=5)

    back_btn_style = BUTTON_STYLE.copy()
    back_btn_style["bg"] = "#777777"
    back_btn_style["activebackground"] = "#555555"

    back_btn = tk.Button(root, text="Return to Command Hub", **back_btn_style, command=show_employer_dashboard)
    back_btn.pack(side="bottom", pady=15)

def show_applicants_screen(job_id, job_title):
    clear_screen()

    title = tk.Label(root, text=f"Applications for:\n{job_title}", font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR)
    title.pack(pady=20)

    candidates = get_job_applicants(job_id)

    if not candidates:
        tk.Label(root, text="No applicants yet.", font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT).pack(pady=50)
    else:
        for candidate in candidates:
            c_box = tk.Frame(root, bg=BACKGROUND_COLOR, bd=1, relief="solid")
            c_box.pack(fill="x", padx=40, pady=8, ipady=6)

            tk.Label(c_box, text=candidate["name"], font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15, pady=2)
            tk.Label(c_box, text=f"Contact Email: {candidate['email']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=MUTED_TEXT).pack(anchor="w", padx=15)
            tk.Label(c_box, text=f"Skills: {candidate['skills']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15, pady=2)

    back_btn_style = BUTTON_STYLE.copy()
    back_btn_style["bg"] = "#777777"
    back_btn_style["activebackground"] = "#555555"

    back_btn = tk.Button(root, text="Return to Active Vacancies Tracking", **back_btn_style, command=show_employer_jobs_screen)
    back_btn.pack(side="bottom", pady=25)

def show_employee_dashboard():
    clear_screen()
    
    session_credentials_package = {
        "id": str(current_user_id),
        "name": current_user_name,
        "role": current_user_role
    }
    
    sub_dashboard_frame = tk.Frame(root, bg=BACKGROUND_COLOR)
    sub_dashboard_frame.pack(fill="both", expand=True)
    
    employee_panel = EmployeeDashboard(master=sub_dashboard_frame, current_user=session_credentials_package)
    employee_panel.pack(fill="both", expand=True)
    
    logout_bar_style = BUTTON_STYLE.copy()
    logout_bar_style["bg"] = ERROR_COLOR
    logout_bar_style["activebackground"] = "#D62828"

    logout_bar = tk.Button(root, text="Logout", **logout_bar_style, command=logout)
    logout_bar.pack(side="bottom", fill="x", pady=5)

logout()
root.mainloop()